# Notebook 06: Optimisation Ladder

**Question:** Where exactly does the damage happen across the optimisation pipeline? Is it ONNX conversion, INT8 quantisation, or both?

**Method:** Plot multiple metrics across all three variants on a single chart — full, onnx_fp32, onnx_int8. Show how each metric evolves as optimisation increases. This surfaces which transition causes the most damage and which metrics are most sensitive.

**Metrics tracked:** F1, recall, precision, false negative rate, ECE, disagreement rate, avg latency, model size.

In [ ]:
import sys
sys.path.append('..')

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import yaml
from pathlib import Path
from quantscope import load_predictions

sns.set_theme(style='whitegrid')

In [ ]:
import os
os.makedirs('../results/charts', exist_ok=True)
print('✅ charts/ folder created')

In [ ]:
# Load all results
data = load_predictions('../results/all_predictions.json')

with open('../results/error_summary.json') as f:
    error_summary = json.load(f)

with open('../results/calibration_summary.json') as f:
    cal_summary = json.load(f)

with open('../results/disagreement_summary.json') as f:
    disagreement = json.load(f)

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

MODEL_SIZES = cfg['model_sizes_mb']

variants = ['full', 'onnx_fp32', 'onnx_int8']
labels   = np.array(data['labels'])
print('Data loaded.')

In [ ]:
# Assemble ladder metrics
ladder = {}

for v in variants:
    es   = error_summary[v]
    lats = data[v]['latencies_ms']

    # Disagreement rate vs full
    if v == 'full':
        disagree_rate = 0.0
    elif v == 'onnx_fp32':
        disagree_rate = len(disagreement['full_vs_fp32']['disagreement_indices']) / len(labels) * 100
    else:
        disagree_rate = len(disagreement['full_vs_int8']['disagreement_indices']) / len(labels) * 100

    # False negative rate = FN / (FN + TP)
    fn_rate = es['fn'] / max(es['fn'] + es['tp'], 1) * 100

    ladder[v] = {
        'f1'             : round(es['f1'] * 100, 2),
        'precision'      : round(es['precision'] * 100, 2),
        'recall'         : round(es['recall'] * 100, 2),
        'fn_rate'        : round(fn_rate, 2),
        'ece'            : round(cal_summary[v]['ece'] * 100, 3),
        'disagree_rate'  : round(disagree_rate, 2),
        'avg_latency_ms' : round(np.mean(lats), 1),
        'size_mb'        : MODEL_SIZES[v],
    }

# Print ladder table
print(f"{'Metric':<20} {'full':>12} {'onnx_fp32':>12} {'onnx_int8':>12}")
print('-' * 60)
metrics = ['f1', 'precision', 'recall', 'fn_rate', 'ece', 'disagree_rate', 'avg_latency_ms', 'size_mb']
units   = ['%', '%', '%', '%', '%', '%', 'ms', 'MB']
for metric, unit in zip(metrics, units):
    print(f"{metric:<20} {str(ladder['full'][metric])+unit:>12} {str(ladder['onnx_fp32'][metric])+unit:>12} {str(ladder['onnx_int8'][metric])+unit:>12}")

In [ ]:
# Chart 5: Optimisation ladder
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Optimisation Ladder: Impact Across the Pipeline', fontsize=14)

x      = np.arange(len(variants))
colors = ['#3b82f6', '#f59e0b', '#ef4444']

plot_metrics = [
    ('f1',            'F1 Score (%)',            True),
    ('recall',        'Recall (%)',               True),
    ('precision',     'Precision (%)',            True),
    ('fn_rate',       'False Negative Rate (%)',  False),
    ('avg_latency_ms','Avg Latency (ms)',          True),
    ('size_mb',       'Model Size (MB)',           True),
]

for ax, (metric, ylabel, lower_is_better_inverse) in zip(axes.flat, plot_metrics):
    vals = [ladder[v][metric] for v in variants]
    bars = ax.bar(x, vals, color=colors, width=0.5, edgecolor='white', linewidth=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(variants, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(ylabel, fontsize=10)

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                str(val), ha='center', va='bottom', fontsize=8)

    # Highlight the INT8 bar if it changed meaningfully
    if abs(vals[2] - vals[0]) > 0.5:
        bars[2].set_edgecolor('#111')
        bars[2].set_linewidth(2)

plt.tight_layout()
plt.savefig('../results/charts/optimisation_ladder.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved optimisation_ladder.png')

In [ ]:
# Save ladder summary
with open('../results/ladder_summary.json', 'w') as f:
    json.dump(ladder, f, indent=2)

print('Saved ladder_summary.json')
print()
print('=== KEY FINDINGS ===')
print(f"ONNX conversion  : zero disagreements, identical metrics — lossless")
print(f"INT8 quantisation: {ladder['onnx_int8']['disagree_rate']}% disagreement rate, "
      f"FN rate {ladder['full']['fn_rate']}% -> {ladder['onnx_int8']['fn_rate']}%, "
      f"{ladder['full']['size_mb']}MB -> {ladder['onnx_int8']['size_mb']}MB "
      f"({round(ladder['full']['size_mb']/ladder['onnx_int8']['size_mb'], 1)}x compression)")